[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/10_gqa.ipynb)

# 🔴 Hard: Grouped Query Attention (GQA)

Implement **Grouped Query Attention** — used in LLaMA 2, Mistral, etc. to reduce KV cache size.

Like MHA, but with **fewer KV heads** than Q heads. Each group of Q heads shares the same K/V head.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA

In [13]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.7 MB/s eta 0:00:00


In [1]:
import torch
import torch.nn as nn
import math

In [18]:
# ✏️ YOUR IMPLEMENTATION HERE

class GroupQueryAttention:
    def __init__(self, d_model, num_heads, num_kv_heads):
        self.q_heads = num_heads
        self.kv_heads = num_kv_heads
        self.group = self.q_heads // self.kv_heads
        self.d_model = d_model
        self.d_k = self.d_model // num_heads
        self.W_q = nn.Linear(self.d_model, self.d_model)
        self.W_k = nn.Linear(self.d_model, self.d_k * self.kv_heads)
        self.W_v = nn.Linear(self.d_model, self.d_k * self.kv_heads)
        self.W_o = nn.Linear(self.d_model, self.d_model)

    def forward(self, x):
        bs, seq, _ = x.shape
        q = self.W_q(x).reshape(bs, seq, self.q_heads, self.d_k).transpose(1,2)
        k = self.W_k(x).reshape(bs, seq, self.kv_heads, self.d_k).transpose(1,2)
        k = k.repeat_interleave(self.group, dim = 1)
        v = self.W_k(x).reshape(bs, seq, self.kv_heads, self.d_k).transpose(1,2)
        v = v.repeat_interleave(self.group, dim = 1)
        attn = torch.softmax(q@k.transpose(-1,-2)/math.sqrt(self.d_k),dim = -1)@v
        attn = attn.transpose(1,2).reshape(bs, seq, -1)
        attn_out = self.W_o(attn)
        return attn_out


In [19]:
# 🧪 Debug
torch.manual_seed(0)
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2)
print("W_q shape:", gqa.W_q.weight.shape)  # (32, 32)
print("W_k shape:", gqa.W_k.weight.shape)  # (8, 32)  — only 2 KV heads * d_k=4

x = torch.randn(2, 6, 32)
out = gqa.forward(x)
print("Output shape:", out.shape)           # (2, 6, 32)

W_q shape: torch.Size([32, 32])
W_k shape: torch.Size([8, 32])
Output shape: torch.Size([2, 6, 32])


In [20]:
from torch_judge import check
check('gqa')


🧪 Testing: Grouped Query Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (6.0ms)
  ✅ [2/5] nn.Linear with correct shapes (0.8ms)
  ✅ [3/5] Degenerates to MHA when kv_heads == heads (1.9ms)
  ✅ [4/5] KV heads are shared correctly (2.8ms)
  ✅ [5/5] Gradient flow (3.3ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (14.7ms total)
  Progress saved. Run status() to see your dashboard.

